# Part 1 — Data Exploration & Integrity

Goal: understand the three datasets, find data quality issues, decide how to handle each one,
and save a clean loan table for Part 2.

Data snapshot: **2025-12-31**. Stated coverage: **Jul 2024 – Dec 2025**.

## 0. Load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

loans_raw = pd.read_csv("loans.csv")
users     = pd.read_csv("users.csv")
merchants = pd.read_csv("merchants.csv")

print(loans_raw.shape, users.shape, merchants.shape)

### Check the date columns
`read_csv` loads dates as plain text by default. Check their type first, then convert.

In [ ]:
print(loans_raw[["origination_date"]].dtypes, "\n")
print(users[["signup_date", "first_loan_date"]].dtypes, "\n")
print(merchants[["integration_date"]].dtypes)

All four are text (`object`/`str`), not dates. Convert them. `errors="coerce"` turns any value that isn't a valid
date into a blank (NaT) instead of crashing, so we can count how many failed.

In [ ]:
loans_raw["origination_date"] = pd.to_datetime(loans_raw["origination_date"], errors="coerce")
users["signup_date"]          = pd.to_datetime(users["signup_date"], errors="coerce")
users["first_loan_date"]      = pd.to_datetime(users["first_loan_date"], errors="coerce")
merchants["integration_date"] = pd.to_datetime(merchants["integration_date"], errors="coerce")

print("Dates that could not be converted:")
print("  loans.origination_date: ", loans_raw["origination_date"].isna().sum())
print("  users.signup_date:      ", users["signup_date"].isna().sum())
print("  users.first_loan_date:  ", users["first_loan_date"].isna().sum())
print("  merchants.integration_date:", merchants["integration_date"].isna().sum())

## 1. Profile each dataset

In [ ]:
loans_raw.head()

In [ ]:
loans_raw.describe()

In [ ]:
# Missing values
loans_raw.isna().sum()

In [ ]:
# Categorical columns
for col in ["product_type", "loan_status", "loan_term_months", "is_new_to_affirm"]:
    print(loans_raw[col].value_counts(), "\n")

In [ ]:
# Does term match product? (Split Pay = 1 month, installments = 3/6/12/18)
pd.crosstab(loans_raw["product_type"], loans_raw["loan_term_months"])

In [ ]:
# APR and MDR ranges by product
loans_raw.groupby("product_type")[["apr", "mdr"]].agg(["min", "max"])

In [ ]:
# Users: numbers, text and dates separately (one combined describe() shows NaN for stats that don't apply)
users.isna().sum()

In [ ]:
users[["total_loans", "total_gmv", "fico_at_first_loan"]].describe()

In [ ]:
users[["state", "age_bucket"]].describe()

In [ ]:
users[["signup_date", "first_loan_date"]].agg(["min", "max"])

In [ ]:
merchants

## 2. Data quality checks

### 2a. Keys

In [ ]:
print("Duplicate loan_id:          ", loans_raw["loan_id"].duplicated().sum())
print("Duplicate user_id:          ", users["user_id"].duplicated().sum())
print("Loans with unknown user:    ", (~loans_raw["user_id"].isin(users["user_id"])).sum())
print("Loans with unknown merchant:", (~loans_raw["merchant_id"].isin(merchants["merchant_id"])).sum())

### 2b. Dates

In [ ]:
in_window = loans_raw["origination_date"].between("2024-07-01", "2025-12-31")
print("Loans outside Jul-2024 to Dec-2025:", (~in_window).sum())
print(loans_raw.loc[~in_window, "origination_date"].dt.year.value_counts())

In [ ]:
# Can a loan be 30/60 days past due before it is 30/60 days old?
# (only in-window loans: ages from the 1923/2031 dates are meaningless)
ok = loans_raw[in_window]
age_days = (pd.Timestamp("2025-12-31") - ok["origination_date"]).dt.days

print("delinquent_30 but loan < 30 days old:      ", ((ok["loan_status"] == "delinquent_30") & (age_days < 30)).sum())
print("delinquent_60 but loan < 60 days old:      ", ((ok["loan_status"] == "delinquent_60") & (age_days < 60)).sum())
print("charged_off within 120 days of origination:", ((ok["loan_status"] == "charged_off") & (age_days < 120)).sum())

### 2c. Values

In [ ]:
L = loans_raw
print("Missing FICO:                       ", L["fico_score"].isna().sum())
print("FICO outside 300-850:               ", (L["fico_score"] < 300).sum() + (L["fico_score"] > 850).sum())
print("Interest-bearing APR outside 10-36%:",
      ((L["product_type"] == "installment_ib") & ~L["apr"].between(0.10, 0.36)).sum())
print("Loan amount <= 0:                   ", (L["loan_amount"] <= 0).sum())

### 2d. Do the money columns add up?
If these don't tie out, every revenue and loss number later would be wrong.

In [ ]:
paid_off    = L["loan_status"] == "paid_off"
charged_off = L["loan_status"] == "charged_off"
unpaid_principal = L["loan_amount"] - (L["total_paid"] - L["interest_collected"])

print("mdr_fees != loan_amount x mdr:                   ",
      ((L["mdr_fees"] - L["loan_amount"] * L["mdr"]).abs() > 0.02).sum())
print("Paid off, but total_paid != amount + interest:   ",
      (paid_off & ((L["total_paid"] - L["loan_amount"] - L["interest_collected"]).abs() > 0.02)).sum())
print("Charged off, but credit_loss != unpaid principal:",
      (charged_off & ((L["credit_loss"] - unpaid_principal).abs() > 0.02)).sum())
print("Credit loss on a paid_off/current loan:          ",
      (L["loan_status"].isin(["paid_off", "current"]) & (L["credit_loss"] > 0)).sum())

current_split = L[(L["product_type"] == "split_pay") & (L["loan_status"] == "current")]
print("'current' Split Pay loans already fully repaid:  ",
      (current_split["total_paid"] >= current_split["loan_amount"]).sum(), "of", len(current_split))

### 2e. Do loans and users agree?

In [ ]:
def compare_to_users(loans):
    per_user = loans.groupby("user_id").agg(n_loans=("loan_id", "count"),
                                           first_loan=("origination_date", "min"))
    u = users.set_index("user_id").join(per_user)
    print("total_loans mismatch:              ", (u["total_loans"] != u["n_loans"]).sum())
    print("first_loan_date mismatch:          ", (u["first_loan_date"] != u["first_loan"]).sum())

    # is_new_to_affirm should be True only on each user's earliest loan
    first_loan_idx = loans.groupby("user_id")["origination_date"].idxmin()
    is_first_loan = loans.index.isin(first_loan_idx)
    print("Flagged new, but not the user's first loan:",
          (loans["is_new_to_affirm"] & ~is_first_loan).sum())
    print("User's first loan, but not flagged new:    ",
          (~loans["is_new_to_affirm"] & is_first_loan).sum())

compare_to_users(loans_raw)

## 3. Data cleaning
Fix what is wrong in the data, and record what is wrong but kept.

### 3.1 Duplicate loans: root cause
We found 160 duplicate `loan_id`s, 160 loans with impossible dates, and users whose totals don't match.
Are these the same problem?

In [ ]:
dups = loans_raw[loans_raw["loan_id"].duplicated(keep=False)].sort_values("loan_id")
dups.head(6)

In [ ]:
# Which columns differ between the two copies of a duplicate loan?
for col in loans_raw.columns:
    n = (dups.groupby("loan_id")[col].nunique() > 1).sum()
    if n > 0:
        print(col, "differs in", n, "pairs")

# Is every out-of-window loan one of these duplicates?
print("All out-of-window loans are duplicates:",
      loans_raw.loc[~in_window, "loan_id"].isin(dups["loan_id"]).all())

**Finding:** the two copies of each duplicate are identical except `origination_date` — one copy has a valid date,
the other is in 1923–1925 or 2031. So the duplicates and the bad dates are the same 160 rows.

**Action:** drop the 160 copies with out-of-window dates. No information is lost because the valid copy remains.

### 3.2 Drop duplicates and re-check

In [ ]:
loans = loans_raw[in_window].copy()
print(len(loans_raw), "->", len(loans), "loans")
print("Duplicate loan_id now:", loans["loan_id"].duplicated().sum())

compare_to_users(loans)   # all mismatches should now be 0

### 3.3 Missing FICO

In [ ]:
# Add merchant fields (vertical, promo program) used from here on
loans = loans.merge(merchants[["merchant_id", "vertical", "has_promo_program"]], on="merchant_id")

In [ ]:
loans["fico_missing"] = loans["fico_score"].isna()

# Where is FICO missing? (overall rate is ~7%)
loans.groupby("vertical")["fico_missing"].mean().round(3)

In [ ]:
# Are loans with missing FICO riskier than loans with FICO?
loans["bad"] = loans["loan_status"].isin(["delinquent_30", "delinquent_60", "charged_off"])

risk = loans.groupby("fico_missing").agg(loans=("loan_id", "count"),
                                         bad_rate=("bad", "mean"),
                                         loss=("credit_loss", "sum"),
                                         amount=("loan_amount", "sum"))
risk["loss_rate"] = risk["loss"] / risk["amount"]
risk[["loans", "bad_rate", "loss_rate"]].round(3)

~7% of loans are missing FICO, spread evenly across verticals. Loans with missing FICO are **not riskier** than the
rest (delinquent-or-worse 4.3% vs 5.7%; loss rate 1.7% vs 2.0%) — slightly lower, but the group is small (483 loans),
so we don't read much into the gap.
**Decision:** keep these loans; label them "Unknown" in FICO cuts instead of guessing a value.

### 3.4 Issues flagged but kept
**Delinquency status earlier than loan age allows (21 loans, 2b).** 10 loans are marked 30 days delinquent before
they are 30 days old, and 11 are marked 60 days delinquent before they are 60 days old. We can't tell which field is
wrong — the origination date (loan age) or the delinquency flag — and it affects only 21 of 7,134 loans (0.3%).
**Decision:** leave them unchanged. Their `credit_loss` dollars are still counted, and in Part 2 we measure risk with
`credit_loss` and "delinquent or worse" rather than the exact days-past-due bucket. (The 29 charge-offs within
120 days can be real — e.g. fraud or first-payment default — so they are not treated as errors.)

**`current` Split Pay loans already fully repaid (134 loans, 2d).** The status says `current` but the payments say
`paid_off`. No loss is involved either way.
**Decision:** leave them unchanged; this doesn't affect loss or revenue numbers.

## 4. Assumptions for the analysis
The data isn't wrong here, but each point changes how we measure things in Part 2.

### 4.1 Young loans haven't had time to default

In [ ]:
loans["orig_quarter"] = loans["origination_date"].dt.to_period("Q")
pct_current = (loans["loan_status"] == "current").groupby(loans["orig_quarter"]).mean() * 100
pct_current.round(0)

In [ ]:
pct_current.plot(kind="bar", color="#2a78d6", title="% of loans still 'current' by origination quarter")
plt.ylabel("%")
plt.show()

Most recent loans are still `current`, so their losses haven't shown up yet. Comparing raw loss rates would make
segments with newer loans look safer than they are.
**Assumption:** in Part 2, compare loss rates by origination quarter, or only on loans originated by Jun-2025.

### 4.2 Credit loss on delinquent loans

In [ ]:
loans["loss_pct"] = loans["credit_loss"] / loans["loan_amount"]
loans.groupby("loan_status")["loss_pct"].mean().round(3)

Delinquent loans already carry a loss before being charged off. How is it calculated?
Compare `credit_loss` to the principal the borrower still owes.

In [ ]:
loans["unpaid_principal"] = loans["loan_amount"] - (loans["total_paid"] - loans["interest_collected"])
loans["loss_to_unpaid"] = loans["credit_loss"] / loans["unpaid_principal"]

bad_loans = loans[loans["loan_status"].isin(["delinquent_30", "delinquent_60", "charged_off"])]
bad_loans.groupby("loan_status")["loss_to_unpaid"].describe()[["count", "25%", "50%", "75%"]].round(2)

In [ ]:
# How much of total loss comes from loans that are only delinquent (not yet charged off)?
loss_by_status = loans.groupby("loan_status")["credit_loss"].sum()
(loss_by_status / loss_by_status.sum()).round(3)

The pattern is clear:
- **charged_off:** loss = 100% of unpaid principal (actual loss)
- **delinquent_60:** loss ≈ 60% of unpaid principal
- **delinquent_30:** loss ≈ 30% of unpaid principal

So for delinquent loans `credit_loss` is an **estimate** — an expected loss (like a loss reserve: roughly 30% of
30-day and 60% of 60-day delinquent loans are expected to charge off) — not a loss that has actually happened,
even though the data dictionary calls it "realized". (Interest-bearing loans sit slightly above 30%/60%, and ~40 loans
are lower at ~12%/~32%; no clear driver.) These estimates are ~32% of all credit loss dollars, so they matter.

**Assumption:** use `credit_loss` as given — i.e. charged-off loss plus an expected loss on delinquent loans.
This is a reasonable forward-looking view. In Part 2 we can show a range: *charged-off only* (low) to
*all delinquent loans charge off* (high).

### 4.3 Cost of funds

In [ ]:
loans.groupby("product_type")["cost_of_funds_bps"].mean().round(0)

Cost of funds ranges from 250 to 450 bps but is about the same for every product (and term, FICO, month).
**Assumption:** it's an annual rate charged on the balance still outstanding. Installment loans are repaid
gradually, so on average about half the principal is outstanding:
funding cost ≈ `loan_amount × bps / 10,000 × term_months / 12 × 0.5` (for Split Pay, use the full amount over 1.5 months
— see 4.4).

### 4.4 Split Pay term
The brief describes Split Pay as 4 payments every 2 weeks (first at checkout, last at week 6),
but every Split Pay loan has `loan_term_months = 1`. Is "1" a real duration or just a label?

In [ ]:
loans["age_days"] = (pd.Timestamp("2025-12-31") - loans["origination_date"]).dt.days

current_split = loans[(loans["product_type"] == "split_pay") & (loans["loan_status"] == "current")]
print("Oldest Split Pay loan still 'current':", current_split["age_days"].max(), "days")

No Split Pay loan is still `current` after ~6 weeks, which fits the product description. So `1` is a **product code**
(the data dictionary says "1 = Split Pay"), not a literal 1-month term.
**Assumption:** use ~6 weeks (1.5 months) for Split Pay when a duration is needed (e.g. cost of funds).

## 5. Clean portfolio at a glance

In [ ]:
monthly_gmv = loans.groupby(loans["origination_date"].dt.to_period("M"))["loan_amount"].sum() / 1000
monthly_gmv.plot(kind="bar", color="#2a78d6", title="Monthly originations ($K)")
plt.show()

print("Loans:", len(loans), "| Users:", loans["user_id"].nunique(), "| GMV: $", round(loans["loan_amount"].sum()))

In [ ]:
# Save clean data for Part 2
loans.to_csv("loans_clean.csv", index=False)

## 6. Summary

**Data cleaning**
| # | Issue | Rows | How handled |
|---|---|---|---|
| 1 | Duplicate `loan_id`s; the extra copy has an impossible date (1923–25, 2031) | 160 | Dropped the bad copy. This also fixed the user-table mismatches and the `is_new_to_affirm` flags |
| 2 | Missing FICO | 483 (~7%) | Not riskier than other loans; kept as "Unknown" FICO band, no imputation |
| 3 | `current` Split Pay loans already fully repaid | 134 | Kept; no impact on losses |
| 4 | Delinquency status earlier than loan age allows | 21 | Kept — can't tell if the date or the flag is wrong, and only 0.3% of loans; use loss $ / "delinquent or worse" instead of exact DPD bucket |

**Assumptions for the analysis**
| # | Topic | Assumption |
|---|---|---|
| 1 | Recent loans haven't had time to default (~25% still current) | Compare losses by origination quarter / seasoned loans only |
| 2 | `credit_loss` on delinquent loans is an estimate (~30% / ~60% of unpaid principal), not realized | Use as given (charged-off + expected loss); show low/high range in Part 2 |
| 3 | Cost of funds doesn't vary by product | Annual rate × term × average balance (½ principal for installments) |
| 4 | Split Pay `loan_term_months = 1` is a product code | Use 1.5 months where duration matters |